# Convergence near a reentrant corner

A finite element method can converge even when the usual order-$h$ energy estimate does not apply. We solve a Poisson problem with a known corner singularity.

In [ ]:
from ngsolve import *
from netgen.occ import WorkPlane, OCCGeometry
from ngsolve.webgui import Draw
import numpy as np
import matplotlib.pyplot as plt

## 1. An L-shaped domain

Remove the lower-right quadrant from the square

$$
\begin{align*}
\Omega=(-1,1)^2\setminus\bigl([0,1]\times[-1,0]\bigr).
\end{align*}
$$

The interior angle at the origin is $\omega=3\pi/2$. In polar coordinates, interior points satisfy $r>0$ and $0<\theta<3\pi/2$, with $\theta$ measured counterclockwise from the positive $x$ axis.

All boundary edges have homogeneous Dirichlet data.

In [ ]:
outline = WorkPlane().MoveTo(0, 0).LineTo(1, 0).LineTo(1, 1).LineTo(-1, 1).LineTo(-1, -1).LineTo(0, -1).Close()
face = outline.Face()
face.edges.name = "boundary"
geometry = OCCGeometry(face, dim=2)

initial_maxh = 0.5
num_levels = 7
quadrature_order = 20
load_bonus_order = 10

view_mesh = Mesh(geometry.GenerateMesh(maxh=initial_maxh))
for _ in range(2):
    view_mesh.Refine()
Draw(view_mesh, name="L-shaped domain");

## 2. A known solution with limited regularity

The corner function

$$
\begin{align*}
s(r,\theta)=r^\beta\sin(\beta\theta),\qquad \beta=\frac{\pi}{\omega}=\frac23,
\end{align*}
$$

is harmonic away from the origin and vanishes on the two sides of the reentrant corner. Multiply it by $w=(1-x^2)(1-y^2)$ to obtain

$$
\begin{align*}
u_{\rm ex}=ws.
\end{align*}
$$

The factor $w$ imposes zero values on the outer edges and equals $1$ at the origin. It therefore preserves the leading singularity. We set

$$
\begin{align*}
f=-\Delta(ws)=-(\Delta w)s-2\nabla w\cdot\nabla s, \qquad \Delta w=-4+2x^2+2y^2.
\end{align*}
$$

For $r>0$,

$$
\begin{align*}
\nabla s = \beta r^{\beta-1}
\begin{pmatrix}
\sin((\beta-1)\theta)\\
\cos((\beta-1)\theta)
\end{pmatrix}.
\end{align*}
$$

Near the corner, $u$ behaves like $r^{2/3}$ and its gradient like $r^{-1/3}$. The gradient is square-integrable, but the second derivatives are not: $u\in H_0^1(\Omega)$ and $u\notin H^2(\Omega)$. More precisely, $u\in H^{1+t}(\Omega)$ for every $0<t<2/3$, but not at $t=2/3$.

In [ ]:
beta = 2/3
r = sqrt(x*x+y*y)
theta = IfPos(-y, atan2(y, x)+2*pi, atan2(y, x))
s = r**beta*sin(beta*theta)
grad_s = beta*r**(beta-1)*CF((sin((beta-1)*theta), cos((beta-1)*theta)))

w = (1-x*x)*(1-y*y)
grad_w = CF((-2*x*(1-y*y), -2*y*(1-x*x)))
u_ex = w*s
grad_u_ex = w*grad_s+s*grad_w
f = (4-2*x*x-2*y*y)*s-2*grad_w*grad_s

Draw(u_ex, view_mesh, name="Exact singular solution", deformation=True, order=3);

## 3. Galerkin method

Let $V_h=S_1^0(\mathcal T_h)\cap H_0^1(\Omega)$. We find $u_h\in V_h$ such that

$$
\begin{align*}
\int_\Omega\nabla u_h\cdot\nabla v_h\,dx=\int_\Omega fv_h\,dx\qquad\forall v_h\in V_h.
\end{align*}
$$

The exact gradient is unbounded at the corner, so we visualize the finite element gradient. Its increasing magnitude near the corner indicates the singularity. Error integration uses the analytic gradient at interior quadrature points.

In [ ]:
def galerkin_solution(mesh):
    fes = H1(mesh, order=1, dirichlet="boundary")
    u, v = fes.TnT()
    a = BilinearForm(grad(u)*grad(v)*dx).Assemble()
    l = LinearForm(f*v*dx(bonus_intorder=load_bonus_order)).Assemble()
    uh = GridFunction(fes)
    uh.vec.data = a.mat.Inverse(fes.FreeDofs(), inverse="sparsecholesky") * l.vec
    return uh


view_uh = galerkin_solution(view_mesh)
Draw(view_uh, view_mesh, name="Galerkin solution", deformation=True)
Draw(Norm(grad(view_uh)), view_mesh, name="Magnitude discrete gradient");

## 4. Error decay under uniform refinement

We compute

$$
\begin{align*}
e_{\mathcal A}(h)=\|\nabla u-\nabla u_h\|_{L^2(\Omega)},\qquad e_0(h)=\|u-u_h\|_{L^2(\Omega)}.
\end{align*}
$$

The first is the energy error, equivalently the $H^1$ seminorm of the error. The full $H^1$ norm is $\sqrt{e_0(h)^2+e_{\mathcal A}(h)^2}$ and has the same asymptotic rate.

For the convergence plots, we use

$$
\begin{align*}
h=\max_{T\in\mathcal T_h}h_T,\qquad h_T=\sqrt{2|T|}.
\end{align*}
$$

On these affine triangles, `specialcf.mesh_size` equals $\sqrt{2|T|}$ and is constant on each element. We interpolate it into `L2(mesh, order=0)` and take the largest coefficient. This measure is equivalent to the largest element diameter on this shape-regular mesh family. Both measures halve under each uniform refinement, so they give the same convergence orders. Both the load and the errors involve nonpolynomial functions, thus their quadrature orders are set explicitly above.

Fractional regularity gives energy bounds with any exponent below $2/3$. The dual Dirichlet problem on this L-shaped domain also has only fractional regularity in general. Thus the duality estimate combines **primal and dual** regularity and gives $L^2$ bounds with any exponent below $4/3$.

For this corner singularity on uniformly refined meshes, $h^{2/3}$ and $h^{4/3}$ are the expected asymptotic reference slopes. Smooth parts of the solution can influence the rates on coarse meshes.

In [ ]:
def mesh_size(mesh):
    h = GridFunction(L2(mesh, order=0))
    h.Interpolate(specialcf.mesh_size)
    return h.vec.FV().NumPy().max()


mesh = Mesh(geometry.GenerateMesh(maxh=initial_maxh))
mesh.Refine()
hs, ndofs, L2_errors, energy_errors = [], [], [], []

with TaskManager():
    for level in range(num_levels):
        uh = galerkin_solution(mesh)
        difference = u_ex-uh
        gradient_difference = grad_u_ex-grad(uh)
        L2_error = sqrt(Integrate(difference**2, mesh, order=quadrature_order))
        energy_error = sqrt(Integrate(gradient_difference**2, mesh, order=quadrature_order))

        hs.append(mesh_size(mesh))
        ndofs.append(uh.space.ndof)
        L2_errors.append(L2_error)
        energy_errors.append(energy_error)

        if level < num_levels-1:
            mesh.Refine()

hs = np.array(hs)
print("level   dofs       h        energy error    L2 error")
for level, (ndof, h, e1, e0) in enumerate(
        zip(ndofs, hs, energy_errors, L2_errors)):
    print(f"{level:3d}  {ndof:7d}  {h:10.4e}  {e1:14.6e}  {e0:12.6e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for ax, values, power, smooth_power, title, label in zip(
        axes, [energy_errors, L2_errors], [2/3, 4/3], [1, 2],
        ["Energy error", "L2 error"], [r"$h^{2/3}$", r"$h^{4/3}$"]):
    ax.loglog(hs, values, "o-", label="Galerkin error")
    ax.loglog(hs, values[-1]*(hs/hs[-1])**power,
              "k--", label="corner reference " + label)
    ax.loglog(hs, values[0]*(hs/hs[0])**smooth_power,
              ":", color="0.5", label=rf"smooth-case slope $h^{smooth_power}$")
    ax.set(xlabel=r"$h=\max_T\sqrt{2|T|}$", ylabel="error", title=title)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
plt.show()

The experimental order of convergence (EOC) is

$$
\begin{align*}
\mathrm{EOC}_i=\frac{\log(e_i/e_{i+1})}{\log(h_i/h_{i+1})}.
\end{align*}
$$

Read the sequence of rates as well as the reference lines. Approaching the asymptotic regime may take several refinements.

In [ ]:
print("mesh pair       energy EOC      L2 EOC")
for i in range(len(hs)-1):
    denominator = np.log(hs[i]/hs[i+1])
    energy_rate = np.log(energy_errors[i]/energy_errors[i+1])/denominator
    L2_rate = np.log(L2_errors[i]/L2_errors[i+1])/denominator
    print(f"{i:2d} -> {i+1:<2d}        {energy_rate:8.3f}      {L2_rate:8.3f}")

## Observe

- Why does continuity of the solution not imply $H^2$ regularity?
- The same $P_1$ method converges on the square and on this L-shaped domain. Which assumption behind the order-$h$ energy estimate is missing here?
- Why does the $L^2$ rate involve the regularity of a second, dual problem?
- Where would smaller elements be most useful? How could concentrating them near the corner change the error for a fixed number of unknowns?

**Takeaway.** Convergence in $H^1$ does not require $H^2$ regularity. Its rate depends on how well the solution can be approximated. The gain in $L^2$ additionally depends on dual regularity. Uniform refinement resolves this corner singularity less efficiently than a smooth solution.

[← Shape regularity](02_shape_regularity.ipynb) · [Inverse estimates →](04_inverse_estimates.ipynb)

[Lecture notebooks](../README.md) · [Lecture notes](../../script/README.md) · [Course overview](../../index.ipynb)